<a href="https://colab.research.google.com/github/dineshaiacademy/5-day-ai-bootcamp/blob/main/Day%201%20-%20LLM%20Fundamentals/Learning/3-%20Prompting%20Fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✍️ Prompting Fundamentals

An LLM only does one thing: reads the text you give it and predicts what comes next. That text is the **prompt** — the only lever you have to control what comes back. This notebook is about pulling that lever well, with a runnable demo for every technique.

> ⚠️ **Runs locally against LM Studio** (`http://localhost:1234/v1`, same setup as notebook 2) — won't run unmodified in Colab.

**You'll do in code:** compare a vague prompt to a clear one, run zero-shot vs. few-shot classification, change an answer's angle with role prompting, ask for checkable step-by-step work, force an exact output shape, and fix the four most common prompting mistakes.

## 📖 What is a prompt?

A **prompt** is the complete text handed to the model in one request — system instructions, prior turns, and your latest message, combined. The model doesn't "understand intent" the way a person does; it computes the next token conditioned on exactly that text, appends it, and repeats. Nothing outside the prompt — not what you *meant*, not what you asked in another chat — affects the output. If it isn't in the prompt, the model can't know it.

## 📖 Why a good prompt matters

A model has no ground truth for "what you actually wanted" — only the prompt text and its training distribution. Where the prompt underspecifies audience, scope, or format, the model fills the gap with the **statistically most common completion pattern** for similar prompts — a guess, not an inference about you. Under-specification doesn't produce "no answer," it produces a **confidently wrong-shaped** one. Tightening the prompt is the only fix, and it works the same way across any model or provider.

## 📖 The four building blocks: Role · Task · Context · Format (RTCF)

Four independent axes of ambiguity — each one you leave unspecified is a degree of freedom the model resolves by guessing:

| Block | Constrains | Example |
|---|---|---|
| 👤 **Role** | Register & depth (vocabulary, assumed background, tone) | "You are a Python teacher." |
| 🎯 **Task** | The action (explain vs. summarize vs. generate vs. classify) | "Explain what a for-loop is." |
| 📖 **Context** | Facts in scope the model can't otherwise know | "The student has never coded." |
| 📦 **Format** | The output's shape (length, structure, exclusions) | "Under 100 words, one example." |

Combined: *"You are a Python teacher. Explain what a for-loop is. The student has never programmed before. Use simple words, stay under 100 words, give one small example."*

You don't need all four in every prompt — a one-line factual question needs none. Use RTCF as a **checklist** when an answer comes back wrong-shaped: identify which axis was left open, then constrain it.

## 📖 Vague vs. clear prompts

"Vague" is measurable: a prompt is vague **in proportion to how many materially different outputs could correctly satisfy it**. "Write about marketing" is satisfied by a tweet, a textbook chapter, or a slide deck. "Write a 3-line Instagram caption for a bakery's new chocolate croissant" is satisfied by a much narrower set. Tightening a prompt means shrinking that set until only outputs you'd actually accept remain.

## ✅ Prerequisites

- [LM Studio](https://lmstudio.ai/) installed, with a chat model downloaded and its local server started (see notebook 2, Step 1, if you need a refresher)
- Python 3.9+ with your bootcamp `venv` activated

## ⚙️ Setup — Connect to Your Local Model

Same boilerplate as notebook 2: install the SDK, point it at LM Studio's local server, and auto-detect whichever chat model is loaded.

In [ ]:
%pip install -q openai

: 

In [ ]:
from openai import OpenAI

BASE_URL = "http://localhost:1234/v1"
client = OpenAI(base_url=BASE_URL, api_key="lm-studio")  # key is required by the SDK but ignored by LM Studio

models = client.models.list()
chat_models = [m.id for m in models.data if "embed" not in m.id.lower()]
if not chat_models:
    raise RuntimeError("No chat model found. Load one in LM Studio's Developer tab and start the server.")

MODEL = chat_models[0]
print(f"✅ Using MODEL: {MODEL}")

In [ ]:
def ask(prompt, system=None, max_tokens=200, temperature=0.0):
    """Send one prompt to the local model and print the reply. Used throughout this notebook."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=temperature,
    )
    print(response.choices[0].message.content.strip())

## 🧪 Demo — Vague vs. Clear, Side by Side

Same model, same underlying request, two prompts. Watch how much more usable the second answer is.

In [ ]:
print("❌ VAGUE PROMPT")
print("-" * 40)
ask("Tell me about Python.")

print("\n✅ CLEAR PROMPT")
print("-" * 40)
ask("Explain Python to a complete beginner in 3 sentences, using one everyday-life analogy.")

## 🧪 Zero-Shot Prompting

Ask the model to do something **without any examples** — it relies purely on its pretrained sense of the task pattern (e.g. "this looks like sentiment classification") to infer format and reasoning. Cheapest to write, and reliable for common task types; it gets shaky for unusual tasks or a very specific output format the model wouldn't guess on its own — the gap few-shot prompting fills next.

In [ ]:
ask(
    "Classify the sentiment of this review as exactly one word, Positive or Negative:\n"
    "'The food was amazing.'"
)

## 🧪 Few-Shot Prompting

Show `k` input→output example pairs before the real input. No weights change — the model performs **in-context pattern completion**: given a sequence of examples, the most probable continuation for a new input matches their style, format, and granularity. This is the reliable way to lock in an **exact output format** (specific labels, a strict schema, a particular tone) that zero-shot might not guess.

In [ ]:
examples_prompt = (
    "Great service -> Positive\n"
    "Terrible food -> Negative\n"
    "Amazing experience -> Positive\n"
    "The delivery was late but the food was amazing -> "
)
ask(examples_prompt, max_tokens=5)

## 👤 Role Prompting

A role/persona instruction (usually in the **system** message, which the model treats as higher-priority framing) shifts the implicit audience and vocabulary — register, assumed background, which facts get foregrounded — without changing the topic. Same question below, three roles, three different answers. It does **not** grant new factual knowledge: "act as a doctor" makes the *style* medical, it doesn't make the model licensed or infallible.

In [ ]:
topic = "Explain what blockchain is."

for role in [
    "You are explaining to a 10-year-old. Use one everyday analogy. Max 3 sentences.",
    "You are explaining to a software engineer. Be technical and precise. Max 3 sentences.",
    "You are a bank auditor. Focus on risk, security, and regulation. Max 3 sentences.",
]:
    print(f"--- {role.split('.')[0]} ---")
    ask(topic, system=role, max_tokens=120)
    print()

## 🧠 Step-by-Step Thinking

Asking for **intermediate, verifiable steps** (a running total, a named sub-result) gives you something you can check line by line — that's the real value, not "the model thinks harder." Ask for the **key steps you'd need to verify the result**, not the model's raw internal reasoning — internal deliberation isn't the same as a clean, auditable explanation.

In [ ]:
ask(
    "A pizza costs $84. Add a 15% tip, then split the total evenly between 3 people. "
    "Show each calculation as a labeled line, then give the final answer per person.",
    max_tokens=200,
)

## 📦 Controlling the Answer's Shape

Format instructions constrain output at the **structural** level, independent of content: count, structure (list/table/JSON), length, exclusions. State counts and limits as hard numbers, not soft language — "a few ideas" can correctly mean 2 or 8; "exactly 5" can't.

In [ ]:
ask(
    "Give me 5 AI startup ideas. Use a numbered list. Keep each idea under 15 words. "
    "Do not give any explanations.",
    max_tokens=150,
)

## 🚫 Common Mistakes — all trace back to a missing RTCF axis

| Mistake | Missing axis | Failure mode |
|---|---|---|
| Too vague ("Write something good.") | Task | Model picks an arbitrary interpretation of "good" |
| Bundling unrelated jobs in one prompt | — | Quality drops on *each* task as effort splits across all of them |
| Missing context ("Write an email to a student.") | Context | Model fills gaps with generic, unpersonalized assumptions |
| No format spec ("Explain AI.") | Format | Output length/structure is whatever's statistically typical, not what you needed |

In [ ]:
print("❌ Too vague")
print("-" * 40)
ask("Write something good.")

print("\n✅ Specific")
print("-" * 40)
ask("Write a friendly 3-line message to a customer apologizing for a late shipment, offering 10% off the next order.")

## 🎯 The One-Line Rule

A good prompt is **WHO + WHAT + KNOW + HOW** (Role + Task + Context + Format). Don't make the model guess — clear prompt → less guessing → better answer.

## 📝 Recap

| Concept | What you learned |
|---|---|
| Prompt | The complete text a model conditions its next-token predictions on — nothing outside it is "known" |
| RTCF | Role, Task, Context, Format — four independent axes of ambiguity you can constrain |
| Vague vs. clear | Vague = many valid outputs could satisfy it; clear = only the one you want can |
| Zero-shot | Ask directly, no examples — relies on pretrained task sense |
| Few-shot | Show `k` example pairs first — locks in an exact format or style |
| Role prompting | Changes audience/depth/angle; doesn't grant new facts or authority |
| Step-by-step | Ask for checkable intermediate steps, not hidden internal reasoning |
| Format control | State counts and limits as hard numbers, not soft language |

**Next:** `llm_fundamentals_multi_provider.ipynb` — these techniques inside a real multi-turn conversation.

## 🏋️ Try It Yourself (optional)

Using the `ask()` helper defined above, rewrite and test each vague prompt below into a clear one using RTCF:

1. `"Write a poem."`
2. `"Help me with my resume."`
3. `"Explain machine learning."`

For each, write your improved prompt in a new code cell, call `ask(...)`, and compare the output to what the vague version would have produced.